# Progetto completo: classificare fiori simulati

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Obiettivi
Realizzare un modello dall’inizio alla previsione e all’esportazione. **Solo libreria standard Python, nessun account, download o GPU**. Dataset sintetico originale AI Lab, 600 esempi di due specie fittizie alfa/beta con lunghezza e larghezza in unità arbitrarie. Distribuzioni parzialmente sovrapposte.

## Protocollo
360 esempi per il training, 120 per scegliere configurazione/soglia e 120 per il test finale. La standardizzazione viene stimata solo sul training. Baseline: classe maggioritaria del training.


In [ ]:
import csv
import json
import math
import random
from pathlib import Path
from statistics import mean, pstdev

rng = random.Random(27)
rows = []
for index in range(600):
    label = index % 2
    length = min(9.5, max(0.5, rng.gauss(3.0 + 3.5 * label, 1.0)))
    width = min(5.0, max(0.2, rng.gauss(1.0 + 1.5 * label, 0.65)))
    rows.append(([length, width], label))
rng.shuffle(rows)
train_rows, valid_rows, test_rows = rows[:360], rows[360:480], rows[480:]
feature_mean = [mean(row[0][j] for row in train_rows) for j in range(2)]
feature_scale = [pstdev(row[0][j] for row in train_rows) for j in range(2)]


def transform(features):
    return [(value - center) / scale for value, center, scale in
            zip(features, feature_mean, feature_scale)]


def sigmoid(value):
    return 1 / (1 + math.exp(-max(-40, min(40, value))))

print('Synthetic examples:', len(rows), 'Features:', ['length', 'width'])


## Addestrare
Usiamo regressione logistica: due pesi e un bias producono una probabilità mediante sigmoid. Minimizza la cross-entropy con discesa del gradiente. Il classificatore è lineare nello spazio delle feature: non è una rete profonda.


In [ ]:
weights = [0.0, 0.0]
bias = 0.0
learning_rate = 0.15
loss_history = []
for epoch in range(350):
    gradient = [0.0, 0.0]
    bias_gradient = 0.0
    epoch_loss = 0.0
    for features, label in train_rows:
        scaled = transform(features)
        probability = sigmoid(sum(w*x for w, x in zip(weights, scaled)) + bias)
        error = probability - label
        for j in range(2):
            gradient[j] += error * scaled[j]
        bias_gradient += error
        clipped = min(1-1e-12, max(1e-12, probability))
        epoch_loss -= label*math.log(clipped) + (1-label)*math.log(1-clipped)
    weights = [w-learning_rate*g/len(train_rows) for w,g in zip(weights,gradient)]
    bias -= learning_rate*bias_gradient/len(train_rows)
    loss_history.append(epoch_loss/len(train_rows))
assert loss_history[-1] < loss_history[0]
print('Cross-entropy:', round(loss_history[0],4), '->', round(loss_history[-1],4))


def predict_probability(features):
    return sigmoid(sum(w*x for w, x in zip(weights, transform(features))) + bias)


def evaluate(data, threshold=0.5):
    matrix = [[0, 0], [0, 0]]
    for features, label in data:
        prediction = int(predict_probability(features) >= threshold)
        matrix[label][prediction] += 1
    accuracy = (matrix[0][0] + matrix[1][1]) / len(data)
    return accuracy, matrix

validation_accuracy, validation_matrix = evaluate(valid_rows)
print('Validation accuracy:', validation_accuracy)
print('Validation confusion matrix:', validation_matrix)


## Valutare una volta sul test
Prima di eseguire la cella, concludi la scelta di learning rate, epoche e soglia. L’assert controlla solo una proprietà attesa del dataset sintetico, non sostituisce l’analisi degli errori.


In [ ]:
# Use the test set only after the modelling decisions are complete.
test_accuracy, test_matrix = evaluate(test_rows)
majority_label = int(mean(label for _, label in train_rows) >= 0.5)
baseline_accuracy = mean(label == majority_label for _, label in test_rows)
print('Baseline test accuracy:', baseline_accuracy)
print('Model test accuracy:', test_accuracy)
print('Test confusion matrix (rows=true, columns=predicted):', test_matrix)
assert test_accuracy > baseline_accuracy
assert test_accuracy >= 0.85
assert all(math.isfinite(value) for value in weights + [bias])


## Utilizzare ed esportare
Salviamo parametri e preprocessing in JSON e i dati sintetici in CSV. Il sito include una piccola interfaccia statica che applica gli stessi pesi in JavaScript, senza server. Rigenerare il modello dal notebook permette di riprodurre il risultato.


In [ ]:
artifact = {
    'model_type': 'logistic_regression',
    'dataset': 'AI Lab synthetic flowers, seed 27, 600 samples',
    'features': ['length', 'width'], 'classes': ['alpha', 'beta'],
    'mean': feature_mean, 'scale': feature_scale,
    'weights': weights, 'bias': bias, 'threshold': 0.5,
    'test_accuracy': test_accuracy, 'baseline_accuracy': baseline_accuracy,
    'test_confusion_matrix': test_matrix,
    'train_size': len(train_rows), 'validation_size': len(valid_rows), 'test_size': len(test_rows),
}
Path('reference-model.json').write_text(json.dumps(artifact, indent=2)+'\n')
with Path('synthetic-flowers.csv').open('w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['length', 'width', 'label'])
    for features, label in rows:
        writer.writerow(features + [label])
print('Saved reference-model.json and synthetic-flowers.csv')
print('New example probability:', predict_probability([5.0, 2.0]))


## Esercizi e miglioramenti
1. Cambia la soglia usando solo la validazione e confronta falsi positivi e negativi.
2. Aumenta il rumore nei dati e registra il cambiamento.
3. Sostituisci i dati sintetici con un dataset documentato e autorizzato.
4. Confronta una baseline e un albero.

## Interfaccia facoltativa con Gradio
Gradio richiede un processo Python, quindi funziona localmente o in un ambiente notebook compatibile; non direttamente su GitHub Pages. Non viene installato automaticamente né presentato come verificato. La funzione da esporre è `predict_probability` con due ingressi numerici.

## Riepilogo
Il progetto comprende dati, split, preprocessing, training, baseline, valutazione, singola previsione ed esportazione. Le prestazioni riguardano solo il problema simulato.
